# Interactive InSAR Source Explorer

Move the sliders to see **instantly** how each earthquake source parameter affects the InSAR displacement field.

| Parameter | Controls |
|-----------|----------|
| **Strike** | Fault orientation (clockwise from North) |
| **Dip** | Fault tilt from horizontal (0° = flat, 90° = vertical) |
| **Rake** | Slip direction: 90° = thrust, −90° = normal, ±180° = right-lateral SS, 0° = left-lateral SS |
| **Depth** | Hypocentre depth — controls spatial spread of deformation |
| **Mw** | Moment magnitude — controls displacement amplitude |

> **Note:** Run locally with `jupyter lab` / `jupyter notebook`, or launch on Colab below.  
> On Google Colab, run the first cell before doing anything else — it enables widget rendering.

In [10]:
# Install dependencies if needed (works on Colab and local environments)
try:
    import eq_insar
    import ipywidgets
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'eq-insar[viz]', 'ipywidgets', '-q'])
    print('Installed — please re-run this cell if imports below fail')

# Enable ipywidgets on Google Colab
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass  # not on Colab

In [11]:
%matplotlib inline

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from eq_insar import generate_synthetic_insar, mw_to_m0, SATELLITES
from eq_insar.core.moment_tensor import fault_dimensions_from_magnitude

plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
print('Ready!')

Ready!


In [12]:
# ── plotting function ───────────────────────────────────────────────────────

def plot_explorer(strike, dip, rake, depth_km, Mw, satellite, orbit):
    grid_extent = max(20.0, depth_km * 7.0)

    r = generate_synthetic_insar(
        Mw=float(Mw),
        strike_deg=float(strike),
        dip_deg=float(dip),
        rake_deg=float(rake),
        depth_km=float(depth_km),
        grid_size=220,
        grid_extent_km=grid_extent,
        satellite=satellite,
        orbit=orbit,
        add_noise=False,
        wrap=True,
        seed=0,
    )
    ext = [r['X_km'].min(), r['X_km'].max(), r['Y_km'].min(), r['Y_km'].max()]

    m0 = mw_to_m0(float(Mw))
    l_km, w_km = fault_dimensions_from_magnitude(float(Mw), 'all')
    peak_los_cm = np.max(np.abs(r['los_displacement'])) * 100

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))

    # ── row 1: 3D displacement components ───────────────────────────────────
    components = [
        ('Ue', 'East displacement (Ue)'),
        ('Un', 'North displacement (Un)'),
        ('Uz', 'Vertical displacement (Uz)'),
    ]
    for ax, (key, title) in zip(axes[0], components):
        data_cm = r[key] * 100
        vmax = max(np.percentile(np.abs(data_cm), 99.5), 0.001)
        im = ax.imshow(data_cm, extent=ext, origin='lower',
                       cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        plt.colorbar(im, ax=ax, label='cm', shrink=0.82, pad=0.02)
        ax.plot(0, 0, 'k*', markersize=10)
        ax.set_title(f'{title}\n(peak {np.max(np.abs(data_cm)):.2f} cm)', fontweight='bold')
        ax.set_xlabel('East (km)')
        ax.set_ylabel('North (km)')

    # ── row 2: InSAR observables ─────────────────────────────────────────────
    # LOS displacement
    los_cm = r['los_displacement'] * 100
    vmax_los = max(np.percentile(np.abs(los_cm), 99.5), 0.001)
    im_los = axes[1, 0].imshow(los_cm, extent=ext, origin='lower',
                                cmap='RdBu_r', vmin=-vmax_los, vmax=vmax_los)
    plt.colorbar(im_los, ax=axes[1, 0], label='cm', shrink=0.82, pad=0.02)
    axes[1, 0].plot(0, 0, 'k*', markersize=10)
    axes[1, 0].set_title(
        f'LOS displacement  [{satellite} {orbit}]\n(peak {peak_los_cm:.2f} cm)',
        fontweight='bold'
    )
    axes[1, 0].set_xlabel('East (km)')
    axes[1, 0].set_ylabel('North (km)')

    # Wrapped phase
    axes[1, 1].imshow(r['phase_wrapped'], extent=ext, origin='lower',
                      cmap='hsv', vmin=-np.pi, vmax=np.pi)
    axes[1, 1].plot(0, 0, 'k*', markersize=10)
    axes[1, 1].set_title('Wrapped phase\n(each colour cycle = 1 fringe)', fontweight='bold')
    axes[1, 1].set_xlabel('East (km)')
    axes[1, 1].set_ylabel('North (km)')

    # hide unused panel
    axes[1, 2].set_visible(False)

    rake_label = {
        90: 'thrust', -90: 'normal',
        180: 'R-lateral SS', -180: 'R-lateral SS', 0: 'L-lateral SS',
    }.get(int(rake), 'oblique')

    fig.suptitle(
        f'Strike={strike}°   Dip={dip}°   Rake={rake}° ({rake_label})   '
        f'Depth={depth_km} km   Mw={Mw:.1f}   M\u2080={m0:.2e} N\u00b7m   '
        f'Fault ~{l_km:.0f}\u00d7{w_km:.0f} km',
        fontsize=12, fontweight='bold', y=1.01,
    )
    plt.tight_layout()
    plt.show()


# ── widgets ─────────────────────────────────────────────────────────────────

SLIDER_LAYOUT = widgets.Layout(width='400px')
STYLE         = {'description_width': '130px'}

w_strike = widgets.IntSlider(
    value=30, min=0, max=360, step=5,
    description='Strike (°)',
    continuous_update=False, style=STYLE, layout=SLIDER_LAYOUT,
)
w_dip = widgets.IntSlider(
    value=45, min=1, max=90, step=5,
    description='Dip (°)',
    continuous_update=False, style=STYLE, layout=SLIDER_LAYOUT,
)
w_rake = widgets.IntSlider(
    value=90, min=-180, max=180, step=5,
    description='Rake (°)',
    continuous_update=False, style=STYLE, layout=SLIDER_LAYOUT,
)
w_depth = widgets.FloatSlider(
    value=10.0, min=1.0, max=30.0, step=1.0,
    description='Depth (km)',
    continuous_update=False, style=STYLE, layout=SLIDER_LAYOUT,
)
w_Mw = widgets.FloatSlider(
    value=6.0, min=4.0, max=7.0, step=0.1,
    description='Magnitude Mw',
    continuous_update=False, readout_format='.1f',
    style=STYLE, layout=SLIDER_LAYOUT,
)
w_sat = widgets.Dropdown(
    options=[
        ('Sentinel-1 (C-band)', 'sentinel1'),
        ('ALOS-2 (L-band)',     'alos2'),
        ('TerraSAR-X (X-band)', 'tsx'),
        ('COSMO-SkyMed (X-band)', 'cosmo_skymed'),
        ('ICEYE (X-band)',       'iceye'),
        ('NISAR (L-band)',       'nisar'),
    ],
    value='sentinel1',
    description='Satellite',
    style=STYLE, layout=SLIDER_LAYOUT,
)
w_orbit = widgets.RadioButtons(
    options=['ascending', 'descending'],
    value='ascending',
    description='Orbit',
    style=STYLE,
    layout=widgets.Layout(width='300px'),
)

# ── preset buttons ───────────────────────────────────────────────────────────

presets = [
    ('Thrust',         dict(dip=45,  rake=90,   tooltip='Thrust (dip=45, rake=90)')),
    ('Normal',         dict(dip=60,  rake=-90,  tooltip='Normal (dip=60, rake=-90)')),
    ('Dip-Slip',       dict(dip=90,  rake=90,   tooltip='Vertical dip-slip (dip=90, rake=90)')),
    ('R-Lateral SS',   dict(dip=90,  rake=180,  tooltip='Right-lateral strike-slip')),
    ('L-Lateral SS',   dict(dip=90,  rake=0,    tooltip='Left-lateral strike-slip')),
    ('Oblique Thrust', dict(dip=45,  rake=135,  tooltip='Oblique thrust + right-lateral')),
]

btn_styles = ['info', 'warning', 'primary', 'success', '', 'danger']
buttons = []
for (label, params), style in zip(presets, btn_styles):
    btn = widgets.Button(
        description=label,
        button_style=style,
        layout=widgets.Layout(width='130px'),
        tooltip=params['tooltip'],
    )
    def _make_cb(p):
        def cb(b):
            w_dip.value  = p['dip']
            w_rake.value = p['rake']
        return cb
    btn.on_click(_make_cb(params))
    buttons.append(btn)

# ── layout & display ─────────────────────────────────────────────────────────

out = widgets.interactive_output(
    plot_explorer,
    {
        'strike':    w_strike,
        'dip':       w_dip,
        'rake':      w_rake,
        'depth_km':  w_depth,
        'Mw':        w_Mw,
        'satellite': w_sat,
        'orbit':     w_orbit,
    },
)

controls = widgets.VBox([
    widgets.HTML('<b style="font-size:13px">Quick presets</b>'),
    widgets.HBox(buttons),
    widgets.HTML('<hr style="margin:8px 0"/>'),
    widgets.HTML('<b style="font-size:13px">Source parameters</b>'),
    w_strike,
    w_dip,
    w_rake,
    w_depth,
    w_Mw,
    widgets.HTML('<hr style="margin:8px 0"/>'),
    widgets.HTML('<b style="font-size:13px">Observation geometry</b>'),
    w_sat,
    w_orbit,
], layout=widgets.Layout(padding='10px', border='1px solid #ddd', border_radius='6px'))

display(widgets.VBox([
    controls,
    widgets.HTML('<div style="height:8px"/>'),
    out,
]))

---
## Tips for exploration

**Fault mechanism quick reference:**

| Rake | Type | InSAR pattern |
|------|------|---------------|
| +90° | Thrust (reverse) | Asymmetric bull's-eye, uplift on hanging wall |
| −90° | Normal | Bull's-eye with subsidence on hanging wall |
| ±180° / 0° | Strike-slip | Symmetric four-lobed butterfly |
| Other | Oblique | Mix of bull's-eye and butterfly |

**Things to try:**
1. Set Dip = 90° and vary Rake from −180° → +180° — watch the pattern transition between all mechanism types.
2. Fix all parameters and vary **Depth** from 2 → 40 km — the signal broadens and weakens dramatically.
3. Fix all parameters and switch **Orbit** ascending ↔ descending — strike-slip patterns change most, thrust/normal least.
4. Switch satellite C-band → L-band → X-band — same displacement, different fringe density.
5. Compare **Thrust** at Mw 5.5 shallow (3 km) vs Mw 6.5 deep (25 km) — similar peak LOS signal, very different spatial extent.

---
*Simulations use the Davis (1986) elastic half-space point source model.*  
*Wavelengths: C-band ~5.6 cm (Sentinel-1), L-band ~23 cm (ALOS-2/NISAR), X-band ~3.1 cm (TSX/COSMO/ICEYE).*